In [1]:
##Librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

##Configurar gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

In [2]:
##Cargar CSVs
envios=pd.read_csv("../data/envios.csv")
incidencias=pd.read_csv("../data/incidencias.csv")
rutas=pd.read_csv("../data/rutas.csv")
vehiculos=pd.read_csv("../data/vehiculos.csv")

print(f"Envios: {envios.shape[0]} filas | {envios.shape[1]} columnas\nIncidencias: {incidencias.shape[0]} filas | {incidencias.shape[1]} columnas\nRutas: {rutas.shape[0]} filas | {rutas.shape[1]} columnas\nVehiculos: {vehiculos.shape[0]} filas | {vehiculos.shape[1]} columnas")

Envios: 1030 filas | 9 columnas
Incidencias: 206 filas | 6 columnas
Rutas: 82 filas | 7 columnas
Vehiculos: 61 filas | 8 columnas


In [3]:
##Información de los CSVs
envios.head()
envios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id_envio       984 non-null    float64
 1   fecha_envio    970 non-null    object 
 2   id_ruta        983 non-null    float64
 3   id_vehiculo    982 non-null    float64
 4   peso_kg        975 non-null    object 
 5   volumen_m3     978 non-null    float64
 6   tipo_carga     976 non-null    object 
 7   estado         980 non-null    object 
 8   fecha_entrega  990 non-null    object 
dtypes: float64(4), object(5)
memory usage: 72.6+ KB


In [4]:
##Ver primeras filas
print("="*50)
print("ENVIOS - Primeras 5 filas")
print("="*50)
envios.head()

ENVIOS - Primeras 5 filas


,id_envio,fecha_envio,id_ruta,id_vehiculo,peso_kg,volumen_m3,tipo_carga,estado,fecha_entrega
0,1.0,01/01/2023,NaN,46.0,12675.8,28.48,Peligrosa,Entregado,05/01/2023
1,2.0,2023-01-01,3.0,27.0,13.6,24.93,Peligrosa,Entregado,05/01/2023
2,3.0,02/01/2023,73.0,18.0,2536.0,50.55,refrigerada,Entregado,2023-01-06
3,4.0,03/01/2023,49.0,45.0,466.0,13.88,PELIGROSA,En tránsito,07/01/2023
4,5.0,2023-01-03,17.0,36.0,4928.9,0.44,Refrigerada,Retrasado,07/01/2023


In [5]:
##Funcion para limpieza de números (valores como ~,$,aprox)
def clean_numeric(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x,(int,float)):
        return float(x)
    x=str(x).replace('~','').replace('$','').replace('aprox','').replace('aproximadamente','').strip()
    try:
        return float(x)
    except:
        return np.nan

Informacion completa de los dataset 

In [6]:
# Hacer una copia del original
envios_limpio = envios.copy()

In [7]:
# 1. Limpiar fechas
envios_limpio['fecha_envio'] = pd.to_datetime(envios_limpio['fecha_envio'], errors='coerce', dayfirst=True)
envios_limpio['fecha_entrega'] = pd.to_datetime(envios_limpio['fecha_entrega'], errors='coerce', dayfirst=True)
# 2. Limpiar columnas numéricas
envios_limpio['peso_kg'] = envios_limpio['peso_kg'].apply(clean_numeric)
envios_limpio['volumen_m3'] = envios_limpio['volumen_m3'].apply(clean_numeric)
# 3.Eliminar filas con id_envio nulo
envios_limpio = envios_limpio.dropna(subset=['id_envio'])
# 4.Eliminar filas sin fechas
envios_limpio = envios_limpio.dropna(subset=['fecha_envio', 'fecha_entrega'])
# 5.Calcular tiempo de entrega
envios_limpio['tiempo_entrega_dias'] = (envios_limpio['fecha_entrega'] - envios_limpio['fecha_envio']).dt.days
# 6.Eliminar valores negativos o extremos
envios_limpio = envios_limpio[envios_limpio['tiempo_entrega_dias'] >= 0]
envios_limpio = envios_limpio[envios_limpio['tiempo_entrega_dias'] <= 30]
# 7.Rellenar nulos restantes
envios_limpio['id_ruta'] = envios_limpio['id_ruta'].fillna(0)
envios_limpio['id_vehiculo'] = envios_limpio['id_vehiculo'].fillna(0)
envios_limpio['peso_kg'] = envios_limpio['peso_kg'].fillna(envios_limpio['peso_kg'].median())
envios_limpio['volumen_m3'] = envios_limpio['volumen_m3'].fillna(envios_limpio['volumen_m3'].median())
envios_limpio['tipo_carga'] = envios_limpio['tipo_carga'].fillna('DESCONOCIDO')
envios_limpio['estado'] = envios_limpio['estado'].fillna('DESCONOCIDO')
# 8. Convertir id_envio a entero
envios_limpio['id_envio'] = envios_limpio['id_envio'].astype(int)

In [8]:
print('-------------------------------')
print("FILAS COMPLETAS\n-------------------------------")
print(f"filas antes de limpiar: {len(envios)}") 
print('\n-------------------------------')
print("LIMPIEZA DE FILAS\n-------------------------------")
# envios_limpio=envios.dropna(subset=['fecha_envio','fecha_entrega'])
print(f'filas despues de limpiar: {len(envios_limpio)}')

-------------------------------
FILAS COMPLETAS
-------------------------------
filas antes de limpiar: 1030

-------------------------------
LIMPIEZA DE FILAS
-------------------------------
filas despues de limpiar: 212


Limpieza de incidencias

In [9]:
##Información de los CSVs
incidencias.info()
incidencias.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_incidencia    193 non-null    float64
 1   id_envio         197 non-null    float64
 2   fecha            199 non-null    object 
 3   tipo_incidencia  200 non-null    object 
 4   descripcion      190 non-null    object 
 5   costo_impacto    198 non-null    object 
dtypes: float64(2), object(4)
memory usage: 9.8+ KB


,id_incidencia,id_envio,fecha,tipo_incidencia,descripcion,costo_impacto
0,1.0,245.0,01/01/2023,Desvío de ruta,Incidencia menor,131322.0
1,2.0,560.0,04/01/2023,Daño en carga,Retraso significativo,1971994.0
2,3.0,301.0,08/01/2023,Robo,Incidencia menor,67912.0
3,4.0,438.0,2023-01-12,Accidente,Requirió asistencia,1326681.0
4,5.0,868.0,2023-01-15,Falla mecánica,Retraso significativo,745599.0


In [10]:
print(f"filas antes de limpiar: {len(incidencias)}") 

filas antes de limpiar: 206


In [11]:
#Crear copia del dataset
incidencias_limpio=incidencias.copy()

In [12]:
#Limpiar fecha
incidencias_limpio['fecha']=pd.to_datetime(incidencias_limpio['fecha'],errors='coerce',dayfirst=True)
incidencias_limpio=incidencias_limpio.dropna(subset=['fecha'])
#Limpiar costo_impacto (valores con $,~,aprox)
incidencias_limpio['costo_impacto']=incidencias_limpio['costo_impacto'].apply(clean_numeric)
incidencias_limpio['costo_impacto']=incidencias_limpio['costo_impacto'].fillna(0)
#Limpiar id_envio (limpieza a los valores NaN)
incidencias_limpio=incidencias_limpio.dropna(subset=['id_envio'])
#Limpiar id_incidencia 
incidencias_limpio=incidencias_limpio.dropna(subset=['id_incidencia'])
#Limpiar tipo_incidencias (NaN='Desconocido')
incidencias_limpio['tipo_incidencia']=incidencias_limpio['tipo_incidencia'].fillna('Desconocido')
#id_envio a dato entero
incidencias_limpio['id_envio']=incidencias_limpio['id_envio'].astype(int)
incidencias_limpio['descripcion']=incidencias_limpio['descripcion'].fillna('Sin descripcion')

In [13]:
##Eliminacion de fechas NaT
print('-------------------------------')
print("FILAS COMPLETAS\n-------------------------------")
print(f"filas antes de limpiar: {len(incidencias)}") 
print('\n-------------------------------')
print("LIMPIEZA DE FILAS\n-------------------------------")
# envios_limpio=envios.dropna(subset=['fecha_envio','fecha_entrega'])
print(f'filas despues de limpiar: {len(incidencias_limpio)}')

-------------------------------
FILAS COMPLETAS
-------------------------------
filas antes de limpiar: 206

-------------------------------
LIMPIEZA DE FILAS
-------------------------------
filas despues de limpiar: 83


In [14]:
incidencias_limpio.head(10)

,id_incidencia,id_envio,fecha,tipo_incidencia,descripcion,costo_impacto
0,1.0,245,2023-01-01,Desvío de ruta,Incidencia menor,131322.0
1,2.0,560,2023-01-04,Daño en carga,Retraso significativo,1971994.0
2,3.0,301,2023-01-08,Robo,Incidencia menor,67912.0
6,7.0,725,2023-01-23,Falla mecánica,Incidencia menor,0.0
12,13.0,942,2023-02-14,Retraso,Sin mayor impacto,563575.0
16,17.0,121,2023-02-28,Retraso,Pérdida parcial,937012.0
17,18.0,88,2023-03-04,Daño en carga,Requirió asistencia,1411124.0
22,23.0,702,2023-03-22,Accidente,Sin descripcion,1137813.0
24,25.0,593,2023-03-30,Robo,Incidencia menor,1411852.0
28,29.0,708,2023-04-13,Accidente,Pérdida parcial,1274515.0


RUTAS

In [15]:
rutas.info()
rutas.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82 entries, 0 to 81
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_ruta              79 non-null     float64
 1   origen               79 non-null     object 
 2   destino              76 non-null     object 
 3   distancia_km         79 non-null     float64
 4   tiempo_estimado_hrs  77 non-null     float64
 5   tipo_via             76 non-null     object 
 6   peaje_total          80 non-null     float64
dtypes: float64(4), object(3)
memory usage: 4.6+ KB


,id_ruta,distancia_km,tiempo_estimado_hrs,peaje_total
count,79.000000,79.000000,77.000000,80.000000
mean,40.493671,1226.345570,11.979221,6874.837500
std,22.913927,2061.669629,7.172652,4459.946612
min,1.000000,29.600000,0.900000,68.000000
25%,21.500000,488.150000,5.200000,2704.750000
50%,40.000000,948.500000,12.000000,6674.000000
75%,59.500000,1552.600000,18.400000,10767.500000
max,80.000000,18627.000000,23.700000,14941.000000


In [16]:
rutas.head(20)

,id_ruta,origen,destino,distancia_km,tiempo_estimado_hrs,tipo_via,peaje_total
0,1.0,La Florida,Pucón,420.3,6.3,Camino Rural,13931.0
1,2.0,NaN,Castro,507.2,19.5,Camino Rural,12065.0
2,3.0,Rancagua,Rengo,679.6,18.4,Urbana,2447.0
3,4.0,Temuco,Santiago,1182.4,9.6,Autopista,3738.0
4,5.0,Cauquenes,San Fernando,914.3,22.9,Camino Rural,5128.0
5,NaN,Castro,Angol,1492.8,21.9,Camino Rural,2524.0
6,7.0,Rengo,Angol,435.0,23.1,Autopista,8725.0
7,8.0,Osorno,Quilpué,31.2,10.6,AUTOPISTA,14632.0
8,9.0,Osorno,Osorno,1555.3,5.1,Ruta Nacional,NaN
9,10.0,Curicó,Curicó,1645.1,15.2,Urbana,6351.0


In [17]:
rutas_limpio=rutas.copy()

In [18]:
#Limpiar columnas numericas
rutas_limpio['distancia_km']=pd.to_numeric(rutas_limpio['distancia_km'],errors='coerce')
rutas_limpio['peaje_total']=pd.to_numeric(rutas_limpio['peaje_total'],errors='coerce')
rutas_limpio['peaje_total']=rutas_limpio['peaje_total'].fillna(0)
rutas_limpio['tiempo_estimado_hrs']=pd.to_numeric(rutas_limpio['tiempo_estimado_hrs'],errors='coerce')
#Limpiar tipo_via (mayúsculas y sin espacios)
rutas_limpio['tipo_via']=rutas_limpio['tipo_via'].str.strip().str.upper()
rutas_limpio['tipo_via']=rutas_limpio['tipo_via'].fillna('DESCONOCIDO')
#Eliminar filas con id_ruta nulo
rutas_limpio=rutas_limpio.dropna(subset=['id_ruta'])
#Limpiar origen y destino en rutas_limpio
rutas_limpio['origen']=rutas_limpio['origen'].fillna('DESCONOCIDO').str.strip().str.upper()
rutas_limpio['destino']=rutas_limpio['destino'].fillna('DESCONOCIDO').str.strip().str.upper()

# Rellenar distancia_km con la mediana
mediana_dist = rutas_limpio['distancia_km'].median()
rutas_limpio['distancia_km'] = rutas_limpio['distancia_km'].fillna(mediana_dist)

# Rellenar tiempo_estimado_hrs con la mediana
mediana_tiempo = rutas_limpio['tiempo_estimado_hrs'].median()
rutas_limpio['tiempo_estimado_hrs'] = rutas_limpio['tiempo_estimado_hrs'].fillna(mediana_tiempo)

In [19]:
##Eliminacion de fechas NaT
print('-------------------------------')
print("FILAS COMPLETAS\n-------------------------------")
print(f"filas antes de limpiar: {len(rutas)}") 
print('\n-------------------------------')
print("LIMPIEZA DE FILAS\n-------------------------------")
# envios_limpio=envios.dropna(subset=['fecha_envio','fecha_entrega'])
print(f'filas despues de limpiar: {len(rutas_limpio)}')

-------------------------------
FILAS COMPLETAS
-------------------------------
filas antes de limpiar: 82

-------------------------------
LIMPIEZA DE FILAS
-------------------------------
filas despues de limpiar: 79


In [20]:
rutas_limpio.head(20)

,id_ruta,origen,destino,distancia_km,tiempo_estimado_hrs,tipo_via,peaje_total
0,1.0,LA FLORIDA,PUCÓN,420.3,6.3,CAMINO RURAL,13931.0
1,2.0,DESCONOCIDO,CASTRO,507.2,19.5,CAMINO RURAL,12065.0
2,3.0,RANCAGUA,RENGO,679.6,18.4,URBANA,2447.0
3,4.0,TEMUCO,SANTIAGO,1182.4,9.6,AUTOPISTA,3738.0
4,5.0,CAUQUENES,SAN FERNANDO,914.3,22.9,CAMINO RURAL,5128.0
6,7.0,RENGO,ANGOL,435.0,23.1,AUTOPISTA,8725.0
7,8.0,OSORNO,QUILPUÉ,31.2,10.6,AUTOPISTA,14632.0
8,9.0,OSORNO,OSORNO,1555.3,5.1,RUTA NACIONAL,0.0
9,10.0,CURICÓ,CURICÓ,1645.1,15.2,URBANA,6351.0
10,11.0,LA SERENA,RENGO,128.4,16.6,URBANA,68.0


VEHICULOS

In [21]:
vehiculos_limpio=vehiculos.copy()
vehiculos.head(10)

,id_vehiculo,placa,tipo,capacidad_kg,capacidad_m3,año_fabricacion,estado_vehiculo,km_recorridos
0,1.0,URLR-78,Camioneta,20000.0,49.2,2018.0,Operativo,267419.0
1,2.0,DPYA-28,Furgón,5000.0,38.7,2019.0,Operativo,179180.0
2,3.0,MAXQ-41,Camión,10000.0,69.4,2012.0,En mantención,478395.0
3,4.0,SDEB-49,Camioneta,1000.0,NaN,2017.0,Operativo,13823.0
4,5.0,VCPV-18,SEMI-REMOLQUE,3000.0,42.4,2010.0,Operativo,154526.0
5,6.0,VCMU-38,Camión,1000.0,74.9,2017.0,Fuera de servicio,145004.0
6,7.0,BFIV-96,NaN,10000.0,48.7,2022.0,Operativo,NaN
7,NaN,KDRO-51,VAN,5000.0,45.4,2016.0,Fuera de servicio,215162.0
8,9.0,VUTP-52,Van,5000.0,39.1,2022.0,Fuera de servicio,77965.0
9,10.0,BHTE-57,Van,1000.0,20.7,2024.0,Operativo,431424.0


In [22]:
#Limpieza id_vehiculo
vehiculos_limpio=vehiculos_limpio[pd.to_numeric(vehiculos_limpio['id_vehiculo'],errors='coerce').notna()]
vehiculos_limpio['id_vehiculo']=vehiculos_limpio['id_vehiculo'].astype(int)
#Limpieza columnas numéricas
vehiculos_limpio['capacidad_kg']=pd.to_numeric(vehiculos_limpio['capacidad_kg'],errors='coerce')
vehiculos_limpio['capacidad_m3']=pd.to_numeric(vehiculos_limpio['capacidad_m3'],errors='coerce')
vehiculos_limpio['km_recorridos']=pd.to_numeric(vehiculos_limpio['km_recorridos'],errors='coerce')
vehiculos_limpio['año_fabricacion']=pd.to_numeric(vehiculos_limpio['año_fabricacion'],errors='coerce')
#Limpieza de columnas categoricas
vehiculos_limpio['tipo']=vehiculos_limpio['tipo'].fillna('DESCONOCIDO').str.strip().str.upper()
vehiculos_limpio['estado_vehiculo']=vehiculos_limpio['estado_vehiculo'].fillna('DESCONOCIDO').str.strip().str.upper()
vehiculos_limpio['placa']=vehiculos_limpio['placa'].fillna('SIN PLACA').str.strip().str.upper()
# Calcular medianas
mediana_cap_kg = vehiculos_limpio['capacidad_kg'].median()
mediana_cap_m3 = vehiculos_limpio['capacidad_m3'].median()
mediana_km = vehiculos_limpio['km_recorridos'].median()
mediana_año = vehiculos_limpio['año_fabricacion'].median()
# Rellenar nulos
vehiculos_limpio['capacidad_kg'] = vehiculos_limpio['capacidad_kg'].fillna(mediana_cap_kg)
vehiculos_limpio['capacidad_m3'] = vehiculos_limpio['capacidad_m3'].fillna(mediana_cap_m3)
vehiculos_limpio['km_recorridos'] = vehiculos_limpio['km_recorridos'].fillna(mediana_km)
vehiculos_limpio['año_fabricacion'] = vehiculos_limpio['año_fabricacion'].fillna(mediana_año)

In [23]:
##Eliminacion de fechas NaT
print('-------------------------------')
print("FILAS COMPLETAS\n-------------------------------")
print(f"filas antes de limpiar: {len(vehiculos)}") 
print('\n-------------------------------')
print("LIMPIEZA DE FILAS\n-------------------------------")
print(f'filas despues de limpiar: {len(vehiculos_limpio)}')

-------------------------------
FILAS COMPLETAS
-------------------------------
filas antes de limpiar: 61

-------------------------------
LIMPIEZA DE FILAS
-------------------------------
filas despues de limpiar: 59


In [24]:
vehiculos_limpio.head(20)

,id_vehiculo,placa,tipo,capacidad_kg,capacidad_m3,año_fabricacion,estado_vehiculo,km_recorridos
0,1,URLR-78,CAMIONETA,20000.0,49.20,2018.0,OPERATIVO,267419.0
1,2,DPYA-28,FURGÓN,5000.0,38.70,2019.0,OPERATIVO,179180.0
2,3,MAXQ-41,CAMIÓN,10000.0,69.40,2012.0,EN MANTENCIÓN,478395.0
3,4,SDEB-49,CAMIONETA,1000.0,37.15,2017.0,OPERATIVO,13823.0
4,5,VCPV-18,SEMI-REMOLQUE,3000.0,42.40,2010.0,OPERATIVO,154526.0
5,6,VCMU-38,CAMIÓN,1000.0,74.90,2017.0,FUERA DE SERVICIO,145004.0
6,7,BFIV-96,DESCONOCIDO,10000.0,48.70,2022.0,OPERATIVO,232542.5
8,9,VUTP-52,VAN,5000.0,39.10,2022.0,FUERA DE SERVICIO,77965.0
9,10,BHTE-57,VAN,1000.0,20.70,2024.0,OPERATIVO,431424.0
10,11,OLYB-22,VAN,20000.0,5.30,2017.0,OPERATIVO,476196.0


In [25]:
#VER DATOS NULOS RESTANTES
print("="*50)
print("VERIFICACIÓN DE VALORES NULOS")
print("="*50)

# 1. Envios
print("\n ENVIOS:")
print(f"  id_envio: {envios_limpio['id_envio'].isna().sum()}")
print(f"  fecha_envio: {envios_limpio['fecha_envio'].isna().sum()}")
print(f"  id_ruta: {envios_limpio['id_ruta'].isna().sum()}")
print(f"  id_vehiculo: {envios_limpio['id_vehiculo'].isna().sum()}")
print(f"  peso_kg: {envios_limpio['peso_kg'].isna().sum()}")
print(f"  volumen_m3: {envios_limpio['volumen_m3'].isna().sum()}")
print(f"  tipo_carga: {envios_limpio['tipo_carga'].isna().sum()}")
print(f"  estado: {envios_limpio['estado'].isna().sum()}")
print(f"  fecha_entrega: {envios_limpio['fecha_entrega'].isna().sum()}")
print(f"  tiempo_entrega_dias nulos: {envios_limpio['tiempo_entrega_dias'].isna().sum()}")
# 2. Incidencias
print("\n INCIDENCIAS:")
print(f"  id_envio: {incidencias_limpio['id_envio'].isna().sum()}")
print(f"  fecha nulos: {incidencias_limpio['fecha'].isna().sum()}")
print(f"  tipo_incidencia nulos: {incidencias_limpio['tipo_incidencia'].isna().sum()}")
print(f'  descripcion: {incidencias_limpio['descripcion'].isna().sum()}') 
print(f"  costo_impacto nulos: {incidencias_limpio['costo_impacto'].isna().sum()}")
# 3. Rutas
print("\n RUTAS:")
print(f"  id_ruta nulos: {rutas_limpio['id_ruta'].isna().sum()}")
print(f"  origen nulos: {rutas_limpio['origen'].isna().sum()}")
print(f"  destino nulos: {rutas_limpio['destino'].isna().sum()}")
print(f"  distancia_km: {rutas_limpio['distancia_km'].isna().sum()}")
print(f"  tiempo_estimado_hrs: {rutas_limpio['tiempo_estimado_hrs'].isna().sum()}")
print(f"  tipo_via nulos: {rutas_limpio['tipo_via'].isna().sum()}")
print(f"  peaje_total nulos: {rutas_limpio['peaje_total'].isna().sum()}")
# 4. Vehiculos
print("\n VEHICULOS:")
print(f"  id_vehiculo nulos: {vehiculos_limpio['id_vehiculo'].isna().sum()}")
print(f"  placa: {vehiculos_limpio['placa'].isna().sum()}")
print(f"  tipo: {vehiculos_limpio['tipo'].isna().sum()}")
print(f"  capacidad_kg nulos: {vehiculos_limpio['capacidad_kg'].isna().sum()}")
print(f"  capacidad_m3 nulos: {vehiculos_limpio['capacidad_m3'].isna().sum()}")
print(f"  año_fabricacion nulos: {vehiculos_limpio['año_fabricacion'].isna().sum()}")
print(f"  estado_vehiculo: {vehiculos_limpio['estado_vehiculo'].isna().sum()}")
print(f"  km_recorridos nulos: {vehiculos_limpio['km_recorridos'].isna().sum()}")

VERIFICACIÓN DE VALORES NULOS

 ENVIOS:
  id_envio: 0
  fecha_envio: 0
  id_ruta: 0
  id_vehiculo: 0
  peso_kg: 0
  volumen_m3: 0
  tipo_carga: 0
  estado: 0
  fecha_entrega: 0
  tiempo_entrega_dias nulos: 0

 INCIDENCIAS:
  id_envio: 0
  fecha nulos: 0
  tipo_incidencia nulos: 0
  descripcion: 0
  costo_impacto nulos: 0

 RUTAS:
  id_ruta nulos: 0
  origen nulos: 0
  destino nulos: 0
  distancia_km: 0
  tiempo_estimado_hrs: 0
  tipo_via nulos: 0
  peaje_total nulos: 0

 VEHICULOS:
  id_vehiculo nulos: 0
  placa: 0
  tipo: 0
  capacidad_kg nulos: 0
  capacidad_m3 nulos: 0
  año_fabricacion nulos: 0
  estado_vehiculo: 0
  km_recorridos nulos: 0


In [1]:
# Si tienes los CSVs originales y ya los limpiaste en este notebook
# Asegúrate de que las variables existan
print(f"envios_limpio existe? {'envios_limpio' in dir()}")
print(f"incidencias_limpio existe? {'incidencias_limpio' in dir()}")
print(f"rutas_limpio existe? {'rutas_limpio' in dir()}")
print(f"vehiculos_limpio existe? {'vehiculos_limpio' in dir()}")

envios_limpio existe? False
incidencias_limpio existe? False
rutas_limpio existe? False
vehiculos_limpio existe? False


In [28]:
envios_limpio.to_csv('../data/envios_limpio.csv', index=False)
incidencias_limpio.to_csv('../data/incidencias_limpio.csv', index=False)
rutas_limpio.to_csv('../data/rutas_limpio.csv', index=False)
vehiculos_limpio.to_csv('../data/vehiculos_limpio.csv', index=False)